# Gated recurrent unit (GRU)
1. The pointwise operations of one GRU step, with the story numbers of the Note.
2. Parameter counts: SimpleRNN, GRU, LSTM.
3. One GRU time step by hand with NumPy, checked against Keras' GRU layer.
4. GRU against LSTM on the long-memory IMDB task of the LSTM Note (same data, same sizes, same training).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
import time
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
try:
    tf.config.experimental.enable_op_determinism()   # same numbers on every run (where the GPU allows it)
except Exception as e:
    print("op determinism not available:", e)
print("GPUs:", tf.config.list_physical_devices("GPU"))
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. One step with the story numbers
Four entries of the hidden state: power, conflict, tragedy, revenge. The gate values are hand-picked.

In [ ]:
h_prev = np.array([0.6, 0.6, 0.7, 0.1])     # memory so far
r = np.array([0.8, 0.2, 0.1, 0.9])          # reset gate
h_cand = np.array([0.7, 0.2, 0.1, 0.2])     # candidate hidden state
z = np.array([0.1, 0.7, 0.8, 0.2])          # update gate
print("r * h_prev            =", np.round(r * h_prev, 2))
print("(1 - z) * h_prev      =", np.round((1 - z) * h_prev, 2))
print("z * h_cand            =", np.round(z * h_cand, 2))
print("h_t                   =", np.round((1 - z) * h_prev + z * h_cand, 2))

## 2. Parameter counts
Input of d = 3 numbers per time step, u = 4 units, as in the Note.

In [ ]:
d, u = 3, 4
def count(layer):
    m = keras.Sequential([keras.Input(shape=(None, d)), layer])
    return m.count_params()
counts = {"SimpleRNN": count(keras.layers.SimpleRNN(u)),
          "GRU (reset_after=False)": count(keras.layers.GRU(u, reset_after=False)),
          "GRU (Keras default, reset_after=True)": count(keras.layers.GRU(u)),
          "LSTM": count(keras.layers.LSTM(u))}
for k, v in counts.items():
    print(f"{k:40} {v}")
assert counts["GRU (reset_after=False)"] == 3 * ((u + d) * u + u)
assert counts["GRU (Keras default, reset_after=True)"] == 3 * ((u + d) * u + 2 * u)
assert counts["LSTM"] == 4 * ((u + d) * u + u)

## 3. A GRU step by hand, checked against Keras
Our formulas (concatenation [h_{t-1}, x_t], row vectors):
r = sigmoid([h, x] W_r + b_r), z = sigmoid([h, x] W_z + b_z),
h_cand = tanh([r * h, x] W_c + b_c), h_new = (1 - z) * h + z * h_cand.
Keras writes h_new = z_k * h + (1 - z_k) * h_cand, so its update gate is our 1 - z.
sigmoid(-a) = 1 - sigmoid(a): giving Keras the negated update-gate weights makes the two identical.

In [ ]:
rng = np.random.default_rng(0)
W_r, W_z, W_c = (rng.normal(0, 0.5, (u + d, u)) for _ in range(3))
b_r, b_z, b_c = (rng.normal(0, 0.1, u) for _ in range(3))
sig = lambda a: 1 / (1 + np.exp(-a))
words = {"cat": [1, 0, 0], "mat": [0, 1, 0], "rat": [0, 0, 1]}
x_seq = np.array([words[w] for w in ("cat", "mat", "rat")], dtype="float32")


def gru_by_hand(x_seq):
    h, states = np.zeros(u), []
    for x in x_seq:
        hx = np.concatenate([h, x])
        r = sig(hx @ W_r + b_r)
        z = sig(hx @ W_z + b_z)
        h_cand = np.tanh(np.concatenate([r * h, x]) @ W_c + b_c)
        h = (1 - z) * h + z * h_cand
        states.append(h)
    return np.array(states)


ours = gru_by_hand(x_seq)
# Keras layout: kernel (d, 3u) and recurrent_kernel (u, 3u), gate order z, r, candidate.
# Our (u + d) x u matrices: first u rows multiply h, last d rows multiply x.
kernel = np.concatenate([-W_z[u:], W_r[u:], W_c[u:]], axis=1)
recurrent = np.concatenate([-W_z[:u], W_r[:u], W_c[:u]], axis=1)
bias = np.concatenate([-b_z, b_r, b_c])
layer = keras.layers.GRU(u, reset_after=False, return_sequences=True)
keras.Sequential([keras.Input(shape=(None, d)), layer])
layer.set_weights([kernel, recurrent, bias])
theirs = layer(x_seq[None]).numpy()[0]
print("by hand:\n", np.round(ours, 4))
print("Keras GRU:\n", np.round(theirs, 4))
assert np.allclose(ours, theirs, atol=1e-5)

Keras' default, reset_after=True, multiplies by the reset gate after the matrix product:
h_cand = tanh(x W_cx + b + r * (h W_ch + b')). Same weights, extra bias b' = 0: the result changes slightly.

In [ ]:
layer_after = keras.layers.GRU(u, return_sequences=True)          # reset_after=True
keras.Sequential([keras.Input(shape=(None, d)), layer_after])
layer_after.set_weights([kernel, recurrent, np.stack([bias, np.zeros(3 * u)])])
after = layer_after(x_seq[None]).numpy()[0]


def gru_reset_after(x_seq):
    h, states = np.zeros(u), []
    for x in x_seq:
        hx = np.concatenate([h, x])
        r = sig(hx @ W_r + b_r)
        z = sig(hx @ W_z + b_z)
        h_cand = np.tanh(x @ W_c[u:] + b_c + r * (h @ W_c[:u]))
        h = (1 - z) * h + z * h_cand
        states.append(h)
    return np.array(states)


assert np.allclose(gru_reset_after(x_seq), after, atol=1e-5)
print("reset_after=True:\n", np.round(after, 4))
print("largest difference from reset_after=False:", np.abs(after - theirs).max().round(4))

## 4. GRU against LSTM on the long-memory task
The setup of the LSTM Note: 5,000 training and 5,000 test reviews, 10,000 most frequent words, the first 50
words of each review, then `gap` padding steps. Embedding of 32, a recurrent layer of 32 units, sigmoid output,
Adam, 8 epochs, batch 64, 3 seeds. Only the recurrent layer (LSTM or GRU) and the gap change.

In [ ]:
VOCAB, KEEP, N = 10000, 50, 5000
(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
x_tr, x_te = x_tr[:N], x_te[:N]
y_tr, y_te = y_tr[:N].astype("float32"), y_te[:N].astype("float32")


def with_gap(x, gap):
    first = keras.utils.pad_sequences(x, maxlen=KEEP, padding="pre", truncating="post")   # first 50 words
    return np.concatenate([first, np.zeros((len(x), gap), dtype="int32")], axis=1).astype("int32")


class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        logs["seconds"] = time.perf_counter() - self.t0


GAPS, SEEDS, EPOCHS = (0, 25), (0, 1, 2), 8
rows, params = [], {}
for gap in GAPS:
    xa, xb = with_gap(x_tr, gap), with_gap(x_te, gap)
    for kind in ("LSTM", "GRU"):
        for seed in SEEDS:
            keras.utils.set_random_seed(seed)
            model = keras.Sequential([keras.Input(shape=(KEEP + gap,)),
                                      keras.layers.Embedding(VOCAB, 32),
                                      getattr(keras.layers, kind)(32),
                                      keras.layers.Dense(1, activation="sigmoid")])
            model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
            params[kind] = model.layers[1].count_params()
            h = model.fit(xa, y_tr, epochs=EPOCHS, batch_size=64, validation_data=(xb, y_te), verbose=0,
                          callbacks=[EpochTimer()])
            for e in range(EPOCHS):
                rows.append(dict(gap=gap, model=kind, seed=seed, epoch=e + 1,
                                 val_accuracy=h.history["val_accuracy"][e], seconds=h.history["seconds"][e]))
            print(f"gap {gap:3d} {kind:4} seed {seed}: final {h.history['val_accuracy'][-1]:.3f}", flush=True)
curves = pd.DataFrame(rows)
curves.round(4).to_csv(DATA / "val_accuracy.csv", index=False)
print("recurrent-layer parameters:", params)

## Results: mean over 3 seeds

In [ ]:
final = curves[curves.epoch == EPOCHS].groupby(["gap", "model"]).val_accuracy.agg(["mean", "min", "max"])
best = curves.groupby(["gap", "model", "seed"]).val_accuracy.max().groupby(["gap", "model"]).mean().rename("best_epoch_mean")
secs = curves[curves.epoch > 1].groupby(["gap", "model"]).seconds.mean().rename("seconds_per_epoch")  # epoch 1 includes set-up
summary = pd.concat([final, best, secs], axis=1).round(3)
summary["parameters"] = [params[m] for m in summary.index.get_level_values("model")]
summary.to_csv(DATA / "summary.csv")
print(summary)
print("chance level:", round(max(y_te.mean(), 1 - y_te.mean()), 3))